In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: A Qiskit Function by Q-CTRL Fire Opal
*[API 참조](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)를 참조하세요*

> **Note:** Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, On-Prem(IBM Quantum Platform API 경유) Plan 사용자에게만 제공되는 실험적 기능입니다. 현재 미리 보기 출시 상태이며, 변경될 수 있습니다.


<Accordion>
<AccordionItem title="Package versions">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하길 권장합니다.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## 개요
Fire Opal Optimization Solver를 사용하면 양자 전문 지식 없이도 양자 하드웨어에서 유틸리티 규모의 최적화 문제를 해결할 수 있습니다. 고수준 문제 정의를 입력하기만 하면 Solver가 나머지를 처리합니다. 전체 워크플로는 노이즈를 인식하며, 내부적으로 [Fire Opal의 성능 관리](/guides/q-ctrl-performance-management)를 활용합니다. Solver는 가장 큰 IBM&reg; QPU에서 전체 디바이스 규모로도 고전적으로 어려운 문제에 대해 지속적으로 정확한 해를 제공합니다.

Solver는 유연하며, 목적 함수 또는 임의의 그래프로 정의된 조합 최적화 문제를 푸는 데 사용할 수 있습니다. 문제를 디바이스 토폴로지에 매핑할 필요가 없습니다. 제약 조건을 페널티 항으로 공식화할 수 있다면 비제약 문제와 제약 문제 모두 풀 수 있습니다. 이 가이드에 포함된 예시는 서로 다른 Solver 입력 유형을 사용하여 비제약 및 제약 유틸리티 규모 최적화 문제를 푸는 방법을 보여줍니다. 첫 번째 예시는 156개 노드의 3-정규 그래프에서 정의된 max-cut 문제이며, 두 번째 예시는 비용 함수로 정의된 50개 노드의 최소 꼭짓점 커버 문제입니다.

Optimization Solver에 대한 접근 권한을 얻으려면 [Q-CTRL에 문의](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com)하세요.
## 함수 설명
Solver는 하드웨어 수준의 오류 억제부터 효율적인 문제 매핑 및 폐루프 고전 최적화까지 전체 알고리즘을 완전히 최적화하고 자동화합니다. 내부적으로 Solver의 파이프라인은 모든 단계에서 오류를 줄여, 의미 있는 확장에 필요한 향상된 성능을 가능하게 합니다. 기반 워크플로는 하이브리드 양자-고전 알고리즘인 QAOA(Quantum Approximate Optimization Algorithm)에서 영감을 받았습니다. 전체 Optimization Solver 워크플로에 대한 자세한 요약은 [게재된 논문](https://arxiv.org/abs/2406.01743)을 참조하세요.

![Optimization Solver 워크플로 시각화](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Optimization Solver로 일반 문제를 풀려면:
1. 문제를 목적 함수, 그래프, 또는 `SparsePauliOp` 스핀 체인으로 정의합니다.
2. Qiskit Functions Catalog를 통해 함수에 연결합니다.
3. Solver로 문제를 실행하고 결과를 가져옵니다.
### 허용된 문제 형식
- 목적 함수의 다항식 표현. Python에서 기존 SymPy Poly 객체로 생성하고 [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr)을 사용하여 문자열로 포맷하는 것이 이상적입니다.
- 특정 문제 유형의 그래프 표현. 그래프는 Python에서 networkx 라이브러리를 사용하여 생성해야 합니다. 그런 다음 networkx 함수 `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`를 사용하여 문자열로 변환해야 합니다.
- 특정 문제의 스핀 체인 표현. 스핀 체인은 `SparsePauliOp` 객체로 표현되어야 합니다. 자세한 내용은 [문서](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp)를 참조하세요.

### 지원되는 Backend
다음 코드를 실행하면 현재 지원되는 Backend 목록을 확인할 수 있습니다. 사용하려는 디바이스가 목록에 없으면 [Q-CTRL에 문의](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com)하여 지원을 추가하세요.

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

## Example: Unconstrained optimization
Run the [maximum cut](https://en.wikipedia.org/wiki/Maximum_cut) (max-cut) problem. The following example demonstrates the Solver's capabilities on a 156-node, 3-regular unweighted graph max-cut problem, but you can also solve weighted graph problems.

In addition to `qiskit-ibm-catalog`, you will also use the following packages to run this example: `networkx` and `numpy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [3]:
# %pip install networkx numpy

## 벤치마크
[게재된 벤치마킹 결과](https://arxiv.org/abs/2406.01743)에 따르면 Solver는 120개 이상의 Qubit을 가진 문제를 성공적으로 해결하며, 심지어 양자 어닐링 및 트랩된 이온 디바이스에 대한 이전에 게재된 결과를 능가합니다. 다음 벤치마크 지표는 몇 가지 예시를 기반으로 문제 유형의 정확도 및 확장성에 대한 대략적인 지표를 제공합니다. 실제 지표는 목적 함수의 항 수(밀도) 및 그 지역성, 변수 수, 다항식 차수 등 다양한 문제 특성에 따라 다를 수 있습니다.

표시된 "Qubit 수"는 엄격한 제한이 아니라 매우 일관된 해 정확도를 기대할 수 있는 대략적인 임계값을 나타냅니다. 더 큰 문제 규모도 성공적으로 해결되었으며, 이러한 한계를 넘어서는 테스트를 권장합니다.

임의의 qubit 연결성은 모든 문제 유형에서 지원됩니다.

| 문제 유형    | qubit 수 | 예시 | 정확도 | 총 시간(초) | 런타임 사용량(초) | 반복 횟수
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| 희소 연결 이차 문제  | 156 | 3-정규 max-cut| 100%     | 1764     | 293          | 16 |
| 고차 이진 최적화 | 156 | Ising 스핀 유리 모델 | 100%      | 1461     | 272          | 16 |
| 밀집 연결 이차 문제 | 50 | 완전 연결 max-cut| 100%      |  1758    | 268  | 12 |
| 페널티 항이 있는 제약 문제 | 50 | 8% 엣지 밀도의 가중 최소 꼭짓점 커버 | 100%      | 1074     | 215 | 10 |
## 시작하기
먼저 [IBM Quantum API 키](http://quantum.cloud.ibm.com/)를 사용하여 인증하세요. 그런 다음 아래와 같이 Qiskit Function을 선택합니다. (이 코드 조각은 이미 [계정을 저장](/guides/functions#install-qiskit-functions-catalog-client)한 것을 가정합니다.)

In [4]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [5]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.svg" alt="Output of the previous code cell" />

### 1. 문제 정의
그래프 문제를 정의하고 `problem_type='maxcut'`을 지정하여 Max-Cut 문제를 실행할 수 있습니다.

In [6]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [8]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. 문제 실행
그래프 기반 입력 방법을 사용할 때는 문제 유형을 지정하세요.

In [9]:
# Get job status
print(maxcut_job.status())

QUEUED


### 3. Retrieve the result
Retrieve the optimal cut value from the results dictionary.

<Admonition type="note">
   The mapping of the variables to the bitstring may have changed. The output dictionary contains a `variables_to_bitstring_index_map` sub-dictionary, which helps to verify the ordering.
</Admonition>

In [10]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


다음과 같이 Qiskit Function 워크로드의 [상태](/guides/functions#check-job-status)를 확인하거나 [결과](/guides/functions#retrieve-results)를 가져올 수 있습니다:

In [11]:
# %pip install numpy networkx sympy

### 1. Define the problem
Define a random MVC problem by generating a graph with randomly weighted nodes.

In [12]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg" alt="Output of the previous code cell" />

### 3. 결과 가져오기
결과 딕셔너리에서 최적 컷 값을 가져옵니다.

> **Note:** 변수와 비트스트링 간의 매핑이 변경되었을 수 있습니다. 출력 딕셔너리에는 순서를 확인하는 데 도움이 되는 `variables_to_bitstring_index_map` 하위 딕셔너리가 포함되어 있습니다.

In [13]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

Now every edge in the graph should include at least one end point from the cover, which can be expressed as the inequality:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Any case where an edge is not connected to the vertex of cover must be penalized. This can be represented in the cost function by adding a penalty of the form $P(1-n_i-n_j+n_i n_j)$ where $P$ is a positive penalty constant. Thus, an unconstrained alternative to the constrained inequality for weighted MVC is:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [14]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

그래프가 밀집 연결되어 있지 않은 경우 [PuLP](https://coin-or.github.io/pulp/)와 같은 오픈 소스 솔버로 문제를 고전적으로 풀어 결과의 정확도를 확인할 수 있습니다. 밀도가 높은 문제는 해를 검증하는 데 고급 고전 솔버가 필요할 수 있습니다.
## 예시: 제약 최적화
이전 max-cut 예시는 일반적인 이차 비제약 이진 최적화 문제입니다. Q-CTRL의 Optimization Solver는 제약 최적화를 포함한 다양한 문제 유형에 사용할 수 있습니다. 제약 조건이 페널티 항으로 모델링된 다항식으로 표현된 문제 정의를 입력하여 임의의 문제 유형을 풀 수 있습니다.

다음 예시는 제약 최적화 문제인 [최소 꼭짓점 커버](https://en.wikipedia.org/wiki/Vertex_cover) (MVC)의 비용 함수를 구성하는 방법을 보여줍니다.
`qiskit-ibm-catalog` 및 `qiskit` 패키지 외에도 이 예시를 실행하려면 `numpy`, `networkx`, `sympy` 패키지가 필요합니다. IPython 커널을 사용하는 노트북에서 이 예시를 실행하는 경우, 다음 셀의 주석을 해제하여 이 패키지들을 설치할 수 있습니다.

In [15]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 1. 문제 정의
무작위로 가중된 노드를 가진 그래프를 생성하여 무작위 MVC 문제를 정의합니다.

In [16]:
print(mvc_job.status())

QUEUED


![이전 코드 셀의 출력](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

가중 MVC의 표준 최적화 모델은 다음과 같이 공식화할 수 있습니다. 먼저, 엣지가 부분 집합의 꼭짓점에 연결되지 않은 경우에 대한 페널티를 추가해야 합니다. 따라서 꼭짓점 $i$가 커버에 있으면(즉, 부분 집합에 있으면) $n_i = 1$, 그렇지 않으면 $n_i = 0$으로 설정합니다. 둘째, 목표는 부분 집합의 꼭짓점 총 수를 최소화하는 것으로, 다음 함수로 나타낼 수 있습니다:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [17]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


이제 그래프의 모든 엣지에는 커버에서 최소 하나의 끝점이 포함되어야 하며, 이는 다음 부등식으로 나타낼 수 있습니다:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

엣지가 커버의 꼭짓점에 연결되지 않은 경우에는 페널티를 부과해야 합니다. 이는 $P$가 양의 페널티 상수인 $P(1-n_i-n_j+n_i n_j)$ 형태의 페널티를 추가하여 비용 함수로 나타낼 수 있습니다. 따라서 가중 MVC의 제약 부등식에 대한 비제약 대안은 다음과 같습니다:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$